# EMS Data Analysis for Simulator

This notebook analyzes historical EMS apparatus data to extract:
1. EMS on-scene time statistics (by incident category)
2. Hospital transport probability (by incident category)
3. Hospital time statistics
4. Hospital locations with coordinates

In [9]:
import pandas as pd
import numpy as np
import os

# Create output directory
os.makedirs('data/ems_stats', exist_ok=True)

In [10]:
# Load EMS apparatus data
ems_df = pd.read_csv('data/Vanderbilt_Fire/DataForVandy011426/ems_apparatus.csv', low_memory=False)
print(f"Loaded {len(ems_df)} EMS records")
ems_df.head()

Loaded 924921 EMS records


,Incident_Number,EMS_Unit_Call_Sign,Incident_PSAP_Call_Date_Time,Incident_Unit_En_Route_Date_Time,Enroute_Scene_Seconds,Incident_Unit_Arrived_On_Scene_Date_Time,ToScene_LeftScene_Seconds,Incident_Unit_Left_Scene_Date_Time,Incident_Patient_Arrived_At_Destination_Date_Time,Destination,LeftScene_Destination_Seconds,Incident_Unit_Back_In_Service_Date_Time,Destination_InService_Seconds,Incident_Latitude,Incident_Longitude,Desination_Latitude,Destination_Longitude
0,FFD190701069636,MED33,2019-07-01 07:30:00,2019-07-01 07:39:45,262.0,2019-07-01 07:44:07,NaN,NaN,NaN,NaN,NaN,2019-07-01 07:52:09,NaN,36.089148,-86.627448,NaN,NaN
1,FFD190701069648,MED11,2019-07-01 08:26:43,2019-07-01 08:29:26,169.0,2019-07-01 08:32:15,356.0,2019-07-01 08:38:56,2019-07-01 08:50:49,Centennial Hospital ED,713.0,2019-07-01 09:16:14,1525.0,36.179878,-86.810580,36.153544,-86.809064
2,FFD190701069658,MED03,2019-07-01 08:52:45,2019-07-01 08:55:10,226.0,2019-07-01 08:58:56,1174.0,2019-07-01 09:18:34,2019-07-01 09:28:00,Skyline Hospital ED,566.0,2019-07-01 09:59:35,1895.0,36.176898,-86.757511,36.245489,-86.749738
3,FFD190701069662,MED28,2019-07-01 09:09:33,2019-07-01 09:10:32,194.0,2019-07-01 09:13:46,NaN,NaN,NaN,NaN,NaN,2019-07-01 09:32:28,NaN,36.170518,-86.678725,NaN,NaN
4,FFD190701069669,MED09,2019-07-01 09:28:30,2019-07-01 09:38:17,281.0,2019-07-01 09:42:58,590.0,2019-07-01 09:52:50,2019-07-01 10:05:04,Vanderbilt Hospital ED,734.0,2019-07-01 10:30:36,1532.0,36.166252,-86.806950,36.144627,-86.801751


In [11]:
# Explore key columns
print("Columns:", ems_df.columns.tolist())
print("\nKey columns for EMS analysis:")
print("- ToScene_LeftScene_Seconds: EMS on-scene time")
print("- LeftScene_Destination_Seconds: Travel time to hospital")
print("- Destination_InService_Seconds: Time at hospital")
print("- Destination: Hospital name")

Columns: ['Incident_Number', 'EMS_Unit_Call_Sign', 'Incident_PSAP_Call_Date_Time', 'Incident_Unit_En_Route_Date_Time', 'Enroute_Scene_Seconds', 'Incident_Unit_Arrived_On_Scene_Date_Time', 'ToScene_LeftScene_Seconds', 'Incident_Unit_Left_Scene_Date_Time', 'Incident_Patient_Arrived_At_Destination_Date_Time', 'Destination', 'LeftScene_Destination_Seconds', 'Incident_Unit_Back_In_Service_Date_Time', 'Destination_InService_Seconds', 'Incident_Latitude', 'Incident_Longitude', 'Desination_Latitude', 'Destination_Longitude']

Key columns for EMS analysis:
- ToScene_LeftScene_Seconds: EMS on-scene time
- LeftScene_Destination_Seconds: Travel time to hospital
- Destination_InService_Seconds: Time at hospital
- Destination: Hospital name


In [12]:
# Load incidents to get category information
incidents_df = pd.read_csv('data/Vanderbilt_Fire/DataForVandy011426/incidents_vandy.csv', low_memory=False)
print(f"Loaded {len(incidents_df)} incidents")

# Check for category column
category_cols = [c for c in incidents_df.columns if 'category' in c.lower() or 'nfpa' in c.lower() or 'type' in c.lower()]
print(f"\nCategory-related columns: {category_cols}")

Loaded 1437452 incidents

Category-related columns: ['NFPACategoryCode', 'NFPACategory', 'IncidentType', 'IncidentTypeCode', 'NFIRSType']


In [13]:
# Merge EMS data with incident categories
# Match on Incident_Number
ems_with_category = ems_df.merge(
    incidents_df[['IncidentNumber', 'NFPACategoryCode', 'NFPACategory', 'IncidentType', 'NFIRSType']],
    left_on='Incident_Number',
    right_on='IncidentNumber',
    how='left'
)
print(f"Merged records: {len(ems_with_category)}")

# Normalize IncidentType to match simulator's incident_type_str format:
# Replace ", " with " -  " and handle "Medical Call" -> "Medical"
def normalize_incident_type(val):
    if pd.isna(val):
        return val
    normalized = val.replace(", ", " -  ")
    if normalized == "Medical Call":
        normalized = "Medical"
    return normalized

ems_with_category['incident_type_key'] = ems_with_category['IncidentType'].apply(normalize_incident_type)
print(f"Unique incident types: {ems_with_category['incident_type_key'].nunique()}")
ems_with_category[['IncidentType', 'incident_type_key', 'NFIRSType']].drop_duplicates().head(20)

Merged records: 924923
Unique incident types: 144


,IncidentType,incident_type_key,NFIRSType
0,Medical Call,Medical,EMS & Rescue
6,Building fire,Building fire,Fire
11,Fires in structure other than in a building,Fires in structure other than in a building,Fire
12,Dispatched and cancelled en route,Dispatched and cancelled en route,Good Intent Call
16,Lock-out,Lock-out,Service Call
28,"Malicious, mischievous false call, other",Malicious - mischievous false call - other,False Alarm False Call
60,No incident found on arrival at dispatch address,No incident found on arrival at dispatch address,Good Intent Call
98,"Good intent call, other",Good intent call - other,Good Intent Call
190,Police matter,Police matter,Service Call
287,"Service call, other",Service call - other,Service Call


## 1. EMS On-Scene Time Analysis

In [14]:
# Filter for valid on-scene times
scene_time_df = ems_with_category[
    (ems_with_category['ToScene_LeftScene_Seconds'].notna()) &
    (ems_with_category['ToScene_LeftScene_Seconds'] > 0) &
    (ems_with_category['ToScene_LeftScene_Seconds'] < 7200)  # Filter outliers > 2 hours
].copy()

print(f"Valid on-scene time records: {len(scene_time_df)}")
print(f"\nOverall on-scene time statistics (seconds):")
print(scene_time_df['ToScene_LeftScene_Seconds'].describe())

Valid on-scene time records: 423775

Overall on-scene time statistics (seconds):
count    423775.000000
mean        624.751684
std         380.182000
min           1.000000
25%         381.000000
50%         562.000000
75%         782.000000
max        7197.000000
Name: ToScene_LeftScene_Seconds, dtype: float64


In [15]:
scene_time_df

,Incident_Number,EMS_Unit_Call_Sign,Incident_PSAP_Call_Date_Time,Incident_Unit_En_Route_Date_Time,Enroute_Scene_Seconds,Incident_Unit_Arrived_On_Scene_Date_Time,ToScene_LeftScene_Seconds,Incident_Unit_Left_Scene_Date_Time,Incident_Patient_Arrived_At_Destination_Date_Time,Destination,...,Incident_Latitude,Incident_Longitude,Desination_Latitude,Destination_Longitude,IncidentNumber,NFPACategoryCode,NFPACategory,IncidentType,NFIRSType,incident_type_key
1,FFD190701069648,MED11,2019-07-01 08:26:43,2019-07-01 08:29:26,169.0,2019-07-01 08:32:15,356.0,2019-07-01 08:38:56,2019-07-01 08:50:49,Centennial Hospital ED,...,36.179878,-86.810580,36.153544,-86.809064,FFD190701069648,25.0,Non-Fire Incidents,Medical Call,EMS & Rescue,Medical
2,FFD190701069658,MED03,2019-07-01 08:52:45,2019-07-01 08:55:10,226.0,2019-07-01 08:58:56,1174.0,2019-07-01 09:18:34,2019-07-01 09:28:00,Skyline Hospital ED,...,36.176898,-86.757511,36.245489,-86.749738,FFD190701069658,25.0,Non-Fire Incidents,Medical Call,EMS & Rescue,Medical
4,FFD190701069669,MED09,2019-07-01 09:28:30,2019-07-01 09:38:17,281.0,2019-07-01 09:42:58,590.0,2019-07-01 09:52:50,2019-07-01 10:05:04,Vanderbilt Hospital ED,...,36.166252,-86.806950,36.144627,-86.801751,FFD190701069669,25.0,Non-Fire Incidents,Medical Call,EMS & Rescue,Medical
7,FFD190701069689,MED23,2019-07-01 10:37:00,2019-07-01 10:38:29,592.0,2019-07-01 10:48:21,742.0,2019-07-01 11:03:22,2019-07-01 11:23:53,West-St. Thomas Hospital,...,36.133704,-86.896978,36.129390,-86.844414,FFD190701069689,25.0,Non-Fire Incidents,Medical Call,EMS & Rescue,Medical
9,FFD190701069691,MED11,2019-07-01 10:39:31,2019-07-01 10:41:33,147.0,2019-07-01 10:44:00,660.0,2019-07-01 10:56:00,2019-07-01 10:59:00,Centennial Hospital ED,...,36.156486,-86.806438,36.153544,-86.809064,FFD190701069691,25.0,Non-Fire Incidents,Medical Call,EMS & Rescue,Medical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924910,FFD260113005543,MED02,2026-01-13 00:00:57,2026-01-13 00:05:17,139.0,2026-01-13 00:07:36,124.0,2026-01-13 00:10:40,2026-01-13 00:18:04,Centennial Hospital ED,...,36.152107,-86.773178,36.153544,-86.809064,FFD260113005543,NaN,NaN,Medical Call,EMS & Rescue,Medical
924912,FFD260113005568,MED11,2026-01-13 01:44:36,2026-01-13 01:49:40,448.0,2026-01-13 01:57:08,892.0,2026-01-13 02:17:00,2026-01-13 02:28:37,Children's Hospital at Vanderbilt ED,...,36.181495,-86.796591,36.139504,-86.804023,FFD260113005568,NaN,NaN,Medical Call,EMS & Rescue,Medical
924913,FFD260113005571,MED19,2026-01-13 01:55:01,2026-01-13 01:59:50,663.0,2026-01-13 02:10:53,710.0,2026-01-13 02:23:50,2026-01-13 02:37:23,Centennial Hospital ED,...,36.217638,-86.817924,36.153544,-86.809064,FFD260113005571,NaN,NaN,Medical Call,EMS & Rescue,Medical
924917,FFD260113005593,MED12,2026-01-13 04:28:32,2026-01-13 04:33:14,275.0,2026-01-13 04:37:49,513.0,2026-01-13 04:48:33,2026-01-13 04:57:33,General Hospital ED,...,36.142502,-86.752700,36.167000,-86.807035,FFD260113005593,NaN,NaN,Medical Call,EMS & Rescue,Medical


In [ ]:
# Calculate on-scene time stats by incident type
scene_time_stats = scene_time_df.groupby('incident_type_key')['ToScene_LeftScene_Seconds'].agg(
    mean='mean',
    variance='var',
    std='std',
    min='min',
    count='count'
).reset_index()

scene_time_stats.columns = ['Category', 'mean', 'variance', 'std', 'min', 'count']
scene_time_stats = scene_time_stats[scene_time_stats['count'] >= 5]  # Filter categories with enough data
print(f"EMS Scene Time Stats by Incident Type ({len(scene_time_stats)} categories):")
scene_time_stats

In [ ]:
# Save scene time stats
scene_time_stats.to_csv('data/ems_stats/scene_time_by_category.csv', index=False)
print("Saved: data/ems_stats/scene_time_by_category.csv")

## 2. Transport Probability Analysis

In [18]:
# A call is considered "transported" if it has a destination
ems_with_category['transported'] = ems_with_category['Destination'].notna() & (ems_with_category['Destination'] != '')

print(f"Overall transport rate: {ems_with_category['transported'].mean():.2%}")
print(f"Transported calls: {ems_with_category['transported'].sum()}")
print(f"Non-transported calls: {(~ems_with_category['transported']).sum()}")

Overall transport rate: 45.62%
Transported calls: 421915
Non-transported calls: 503008


In [19]:
# Calculate transport probability by incident type
transport_stats = ems_with_category.groupby('incident_type_key').agg(
    total_calls=('transported', 'count'),
    transported_calls=('transported', 'sum')
).reset_index()

transport_stats['transport_probability'] = transport_stats['transported_calls'] / transport_stats['total_calls']
transport_stats = transport_stats[transport_stats['total_calls'] >= 30]  # Filter categories with enough data
transport_stats.columns = ['Category', 'count', 'transported_count', 'transport_probability']

print(f"Transport Probability by Incident Type ({len(transport_stats)} categories):")
transport_stats

Transport Probability by Incident Type (99 categories):


,Category,count,transported_count,transport_probability
0,Accident - potential accident - other,70,11,0.157143
1,Accidental medical alarm activation - No assis...,1723,17,0.009867
2,Accidental smart device crash/fall activation ...,500,14,0.028000
3,Active Shooter,173,17,0.098266
7,Alarm system activation - no fire - unintenti...,628,5,0.007962
...,...,...,...,...
137,Vicinity alarm (incident in other location) - ...,85,4,0.047059
139,Water or steam leak,107,1,0.009346
140,Water problem - other,80,2,0.025000
142,Wind storm - tornado/hurricane assessment,31,4,0.129032


In [ ]:
# Save transport stats
transport_stats[['Category', 'transport_probability', 'count']].to_csv(
    'data/ems_stats/transport_prob_by_category.csv', index=False
)
print("Saved: data/ems_stats/transport_prob_by_category.csv")

## 3. Hospital Time Analysis

In [21]:
# Filter for records with hospital time (Destination_InService_Seconds)
hospital_time_df = ems_with_category[
    (ems_with_category['Destination_InService_Seconds'].notna()) &
    (ems_with_category['Destination_InService_Seconds'] > 0) &
    (ems_with_category['Destination_InService_Seconds'] < 10800)  # Filter outliers > 3 hours
].copy()

print(f"Valid hospital time records: {len(hospital_time_df)}")
print(f"\nHospital time statistics (seconds):")
print(hospital_time_df['Destination_InService_Seconds'].describe())
print(f"\nMean hospital time: {hospital_time_df['Destination_InService_Seconds'].mean()/60:.1f} minutes")

Valid hospital time records: 422117

Hospital time statistics (seconds):
count    422117.000000
mean       1270.710400
std         601.458474
min           1.000000
25%         878.000000
50%        1238.000000
75%        1545.000000
max       10785.000000
Name: Destination_InService_Seconds, dtype: float64

Mean hospital time: 21.2 minutes


In [22]:
# Calculate hospital time stats (overall and by hospital)
hospital_time_overall = pd.DataFrame({
    'Category': ['Overall'],
    'mean': [hospital_time_df['Destination_InService_Seconds'].mean()],
    'variance': [hospital_time_df['Destination_InService_Seconds'].var()],
    'std': [hospital_time_df['Destination_InService_Seconds'].std()],
    'count': [len(hospital_time_df)]
})

# Also by hospital
hospital_time_by_dest = hospital_time_df.groupby('Destination')['Destination_InService_Seconds'].agg(
    mean='mean',
    variance='var',
    std='std',
    count='count'
).reset_index()
hospital_time_by_dest.columns = ['Hospital', 'mean', 'variance', 'std', 'count']
hospital_time_by_dest = hospital_time_by_dest[hospital_time_by_dest['count'] >= 50]

print("Hospital Time by Destination:")
hospital_time_by_dest.sort_values('count', ascending=False)

Hospital Time by Destination:


,Hospital,mean,variance,std,count
13,Skyline Hospital ED,1438.951255,4.728625e+05,687.649990,69812
10,Midtown-St Thomas Hospital,1209.924554,2.599015e+05,509.805370,63105
2,Centennial Hospital ED,1304.880704,3.552790e+05,596.052880,55878
20,Vanderbilt Hospital ED,1325.646652,3.952678e+05,628.703288,46832
14,Southern Hills Hospital ED,1211.543303,2.956953e+05,543.778753,46348
17,Summit Hospital ED,1336.814408,3.733408e+05,611.016225,43671
23,West-St. Thomas Hospital,1334.938127,2.841443e+05,533.051906,32373
7,General Hospital ED,962.668170,2.038494e+05,451.496841,28602
6,Children's Hospital at Vanderbilt ED,1255.657752,3.908204e+05,625.156308,13499
22,Veterans Administration Hospital ED,885.687833,1.768675e+05,420.556143,6189


In [ ]:
# Save hospital time stats
hospital_time_overall.to_csv('data/ems_stats/hospital_turnaround_overall.csv', index=False)
hospital_time_by_dest.to_csv('data/ems_stats/hospital_turnaround_by_dest.csv', index=False)
print("Saved: data/ems_stats/hospital_turnaround_overall.csv")
print("Saved: data/ems_stats/hospital_turnaround_by_dest.csv")

## 4. Hospital Locations

In [24]:
# Extract unique hospitals with their coordinates
hospital_df = ems_with_category[
    (ems_with_category['Destination'].notna()) &
    (ems_with_category['Destination'] != '') &
    (ems_with_category['Desination_Latitude'].notna()) &
    (ems_with_category['Destination_Longitude'].notna())
][['Destination', 'Desination_Latitude', 'Destination_Longitude']].copy()

# Get unique hospitals with their most common coordinates
hospitals = hospital_df.groupby('Destination').agg(
    lat=('Desination_Latitude', 'median'),
    lon=('Destination_Longitude', 'median'),
    count=('Destination', 'count')
).reset_index()

hospitals.columns = ['Name', 'lat', 'lon', 'visit_count']
hospitals = hospitals.sort_values('visit_count', ascending=False).reset_index(drop=True)
hospitals['Index'] = hospitals.index
hospitals['ID'] = hospitals['Name'].str.replace(' ', '_').str.replace(r'[^a-zA-Z0-9_]', '', regex=True)

print(f"Found {len(hospitals)} unique hospitals")
hospitals

Found 24 unique hospitals


,Name,lat,lon,visit_count,Index,ID
0,Skyline Hospital ED,36.245489,-86.749738,69987,0,Skyline_Hospital_ED
1,Midtown-St Thomas Hospital,36.153889,-86.802386,63245,1,MidtownSt_Thomas_Hospital
2,Centennial Hospital ED,36.153544,-86.809064,56022,2,Centennial_Hospital_ED
3,Vanderbilt Hospital ED,36.144627,-86.801751,46959,3,Vanderbilt_Hospital_ED
4,Southern Hills Hospital ED,36.076451,-86.721460,46441,4,Southern_Hills_Hospital_ED
5,Summit Hospital ED,36.176298,-86.608415,43740,5,Summit_Hospital_ED
6,West-St. Thomas Hospital,36.129390,-86.844414,32424,6,WestSt_Thomas_Hospital
7,General Hospital ED,36.167000,-86.807035,28702,7,General_Hospital_ED
8,Children's Hospital at Vanderbilt ED,36.139504,-86.804023,13553,8,Childrens_Hospital_at_Vanderbilt_ED
9,Veterans Administration Hospital ED,36.141755,-86.803758,6211,9,Veterans_Administration_Hospital_ED


In [ ]:
# Save hospital locations
hospitals_export = hospitals[['Index', 'ID', 'Name', 'lat', 'lon', 'visit_count']]
hospitals_export.to_csv('data/ems_stats/hospital_locations.csv', index=False)
print("Saved: data/ems_stats/hospital_locations.csv")
print(f"\nTop 10 hospitals by visit count:")
hospitals_export.head(10)

## 5. Summary Statistics

In [ ]:
print("=" * 60)
print("EMS DATA ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTotal EMS records analyzed: {len(ems_df)}")
print(f"\n1. EMS ON-SCENE TIME:")
print(f"   Mean: {scene_time_df['ToScene_LeftScene_Seconds'].mean():.0f} seconds ({scene_time_df['ToScene_LeftScene_Seconds'].mean()/60:.1f} min)")
print(f"   Std:  {scene_time_df['ToScene_LeftScene_Seconds'].std():.0f} seconds")

print(f"\n2. TRANSPORT PROBABILITY:")
print(f"   Overall: {ems_with_category['transported'].mean():.1%}")

print(f"\n3. HOSPITAL TIME:")
print(f"   Mean: {hospital_time_df['Destination_InService_Seconds'].mean():.0f} seconds ({hospital_time_df['Destination_InService_Seconds'].mean()/60:.1f} min)")
print(f"   Std:  {hospital_time_df['Destination_InService_Seconds'].std():.0f} seconds")

print(f"\n4. HOSPITALS:")
print(f"   Unique hospitals: {len(hospitals)}")

print("\n" + "=" * 60)
print("OUTPUT FILES:")
print("=" * 60)
print("  data/ems_stats/scene_time_by_category.csv")
print("  data/ems_stats/transport_prob_by_category.csv")
print("  data/ems_stats/hospital_turnaround_overall.csv")
print("  data/ems_stats/hospital_turnaround_by_dest.csv")
print("  data/ems_stats/hospital_locations.csv")